## Phase 3, Attempt 1: class_weight='balanced'

The baseline treated every row equally during training, even though
legit outnumbers fraud ~578:1. `class_weight='balanced'` reweights
the loss so each class contributes equally in aggregate, fraud rows
get upweighted ~578x, legit rows get downweighted slightly.

Everything else (split, features, threshold) stays identical to the
baseline, so any change in the metrics is attributable to this one
lever. Expectation: recall up, precision down, the classic tradeoff,
made visible.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    average_precision_score,
)

df = pd.read_csv("data/raw/creditcard.csv")

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())  # sanity check: should both be ~0.0017, matching baseline

(227845, 30) (56962, 30)
0.001729245759178389 0.0017204452090867595


In [5]:
model = LogisticRegression(max_iter=5000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_scores = model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
auprc = average_precision_score(y_test, y_scores)

print(f"Precision: {precision:.4f}  (should match baseline: 0.8312)")
print(f"Recall:    {recall:.4f}  (should match baseline: 0.6531)")
print(f"AUPRC:     {auprc:.4f}  (should match baseline: 0.7427)")

Precision: 0.8312  (should match baseline: 0.8312)
Recall:    0.6531  (should match baseline: 0.6531)
AUPRC:     0.7427  (should match baseline: 0.7427)


In [3]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
model_balanced.fit(X_train, y_train)

y_pred_balanced = model_balanced.predict(X_test)
y_scores_balanced = model_balanced.predict_proba(X_test)[:, 1]

cm_balanced = confusion_matrix(y_test, y_pred_balanced)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_balanced)

precision_balanced = precision_score(y_test, y_pred_balanced)
recall_balanced = recall_score(y_test, y_pred_balanced)
auprc_balanced = average_precision_score(y_test, y_scores_balanced)

print(f"\nPrecision: {precision_balanced:.4f}  (baseline was 0.8312)")
print(f"Recall:    {recall_balanced:.4f}  (baseline was 0.6531)")
print(f"AUPRC:     {auprc_balanced:.4f}  (baseline was 0.7427)")

Confusion matrix [[TN, FP], [FN, TP]]:
[[55485  1379]
 [    8    90]]

Precision: 0.0613  (baseline was 0.8312)
Recall:    0.9184  (baseline was 0.6531)
AUPRC:     0.7159  (baseline was 0.7427)


## Phase 3, Attempt 1 result: class_weight='balanced'

| Metric    | Baseline | Balanced |
|-----------|----------|----------|
| Precision | 0.8312   | 0.0613   |
| Recall    | 0.6531   | 0.9184   |
| AUPRC     | 0.7427   | 0.7159   |

Confusion matrix: TP=90, FN=8, FP=1379, TN=55,485 (catches 90 of 98 frauds).

class_weight='balanced' massively improved recall (catches nearly all
fraud) but destroyed precision (1 in 40 legit transactions now falsely
flagged) likely unusable in practice as-is.

Notably, AUPRC barely changed (0.74 → 0.72). This means the model's
underlying ability to *rank* transactions by suspicion didn't really
improve  class_weight mainly shifted where the default 0.5 threshold
falls relative to that ranking, not the ranking itself. This suggests
threshold tuning, not just reweighting, is the next lever to pull.

## Phase 3, Attempt 2: threshold tuning

predict() applies a fixed 0.5 cutoff to the probability scores.
That cutoff is a default, not a requirement. We can apply any
threshold we want to y_scores_balanced ourselves, trading recall
against precision along the way.

Goal: find a threshold that keeps recall high (catching most fraud)
without precision collapsing to 0.06 the way it did at 0.5.

In [4]:
import numpy as np

thresholds = np.arange(0.1, 0.95, 0.05)

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
for t in thresholds:
    y_pred_t = (y_scores_balanced >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f}")

 Threshold  Precision     Recall
      0.10     0.0081     0.9490
      0.15     0.0119     0.9490
      0.20     0.0164     0.9388
      0.25     0.0214     0.9184
      0.30     0.0275     0.9184
      0.35     0.0344     0.9184
      0.40     0.0424     0.9184
      0.45     0.0508     0.9184
      0.50     0.0613     0.9184
      0.55     0.0731     0.9184
      0.60     0.0878     0.9082
      0.65     0.1028     0.9082
      0.70     0.1213     0.9082
      0.75     0.1399     0.8980
      0.80     0.1606     0.8980
      0.85     0.1855     0.8878
      0.90     0.2437     0.8878


## Phase 3, Attempt 2 result: threshold tuning on the balanced model

Precision barely improves even at threshold 0.90 (only reaching 0.24),
while recall stays high across nearly the whole range. This suggests
class_weight='balanced' distorted the model's probability calibration,
inflating scores broadly rather than reflecting true confidence. That
makes the threshold behave poorly here, it can't cleanly separate
confident from unconfident predictions.

In [6]:
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
for t in thresholds:
    y_pred_t = (y_scores >= t).astype(int)
    p = precision_score(y_test, y_pred_t, zero_division=0)
    r = recall_score(y_test, y_pred_t, zero_division=0)
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f}")

 Threshold  Precision     Recall
      0.10     0.7037     0.7755
      0.15     0.7212     0.7653
      0.20     0.7327     0.7551
      0.25     0.7245     0.7245
      0.30     0.7419     0.7041
      0.35     0.7614     0.6837
      0.40     0.8025     0.6633
      0.45     0.8205     0.6531
      0.50     0.8312     0.6531
      0.55     0.8289     0.6429
      0.60     0.8267     0.6327
      0.65     0.8267     0.6327
      0.70     0.8429     0.6020
      0.75     0.8358     0.5714
      0.80     0.8281     0.5408
      0.85     0.8226     0.5204
      0.90     0.8197     0.5102


## Phase 3, Attempt 3 result: threshold tuning on the baseline model

Unlike the balanced model, the baseline's threshold sweep behaves
exactly as expected: precision rises steadily (0.70 to 0.84) as
recall falls steadily (0.78 to 0.51) across the range 0.10 to 0.90.
This confirms the baseline's probabilities are well calibrated,
in contrast to class_weight='balanced', which distorted them.

Chosen threshold: 0.20, giving precision 0.7327 and recall 0.7551.
In fraud detection, missing real fraud (a false negative) tends to
cost more than a false alarm (a false positive a human can quickly
review and clear), so recall is weighted more heavily here. This
threshold catches noticeably more fraud than the default 0.5 cutoff
(recall 0.76 vs 0.65) while keeping precision at a still reasonable
0.73, roughly 3 out of 4 flags are real fraud.

This threshold sweep, on its own, already beats the original baseline's
recall (0.65) without needing class_weight at all, just by moving the
decision boundary. However it's still a fundamentally imbalanced-trained
model, we haven't yet corrected the training itself.

## Phase 3, Attempt 4: SMOTE (Synthetic Minority Oversampling)

class_weight reweighted the loss function but left the training data
itself just as imbalanced (~394 fraud rows out of ~227,845). SMOTE
instead generates new synthetic fraud examples by interpolating
between existing fraud rows' nearest neighbors, until the training
set is roughly balanced.

Critical rule: SMOTE is applied only to the training set, never the
test set, the test set must stay untouched and representative of
real world data, or evaluation becomes meaningless.

In [7]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:", y_train_smote.value_counts().to_dict())

Before SMOTE: {0: 227451, 1: 394}
After SMOTE: {0: 227451, 1: 227451}


## Phase 3, training on SMOTE-resampled data

Training a fresh logistic regression on X_train_smote / y_train_smote
(the balanced, synthetic-augmented training set), then evaluating on
the original, untouched X_test / y_test, same as every prior attempt.
This keeps the comparison fair: same test set throughout, only the
training data and method change.

In [9]:
model_smote = LogisticRegression(max_iter=10000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = model_smote.predict(X_test)
y_scores_smote = model_smote.predict_proba(X_test)[:, 1]

cm_smote = confusion_matrix(y_test, y_pred_smote)
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm_smote)

precision_smote = precision_score(y_test, y_pred_smote)
recall_smote = recall_score(y_test, y_pred_smote)
auprc_smote = average_precision_score(y_test, y_scores_smote)

print(f"\nPrecision: {precision_smote:.4f}  (baseline 0.8312, balanced 0.0613)")
print(f"Recall:    {recall_smote:.4f}  (baseline 0.6531, balanced 0.9184)")
print(f"AUPRC:     {auprc_smote:.4f}  (baseline 0.7427, balanced 0.7159)")

Confusion matrix [[TN, FP], [FN, TP]]:
[[56309   555]
 [   10    88]]

Precision: 0.1369  (baseline 0.8312, balanced 0.0613)
Recall:    0.8980  (baseline 0.6531, balanced 0.9184)
AUPRC:     0.7389  (baseline 0.7427, balanced 0.7159)


## Phase 3 result: SMOTE

| Metric    | Baseline | Balanced | SMOTE  |
|-----------|----------|----------|--------|
| Precision | 0.8312   | 0.0613   | 0.1369 |
| Recall    | 0.6531   | 0.9184   | 0.8980 |
| AUPRC     | 0.7427   | 0.7159   | 0.7389 |

Confusion matrix: TP=88, FN=10, FP=555, TN=56,309.

SMOTE lands between class_weight and the baseline: fewer false alarms
than class_weight (555 vs 1379), but still far more than the baseline
(13). Recall improves sharply (0.65 to 0.90), similar to class_weight.

Critically, AUPRC (0.7389) is nearly identical to the baseline's
(0.7427), much closer than class_weight's (0.7159). This suggests
SMOTE, like class_weight, mainly shifted the model's calibration
(pushing probabilities higher because it trained on artificially
balanced data) rather than improving its actual ability to rank
transactions by suspicion. The underlying skill ceiling for logistic
regression here appears to be around AUPRC ~0.74, regardless of how
the imbalance is corrected.

Next: try threshold tuning on this SMOTE model too, since like the
balanced model, its calibration may be shifted, and a non-default
threshold might recover a usable precision/recall tradeoff.